# Optimización de Crop3Test3

100 pruebas sobre 1000 mezclas fijas de entrenamiento, maximizando SSIM. Se buscan máscara, correcciones, saturación local, confianza, muestreo latente. alpha_mix=0.5 y máximo 20 iteraciones quedan fijos. Test se reserva para evaluar el ganador en tres corridas; no interviene en la búsqueda.


In [ ]:
import importlib
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
import optuna
import matplotlib.pyplot as plt

project_root = next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "project" / "CROP" / "models").is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from project.loader import Loader
import project.CROP.models.crop3_test as crop3_test_module
import project.CROP.models.crop3_test3 as crop3_test3_module
importlib.reload(crop3_test_module)
importlib.reload(crop3_test3_module)

SEED = 3333
DATASET = "fashion"
CVAE_NAME = "int_256_lat_64_beta_0_4_fashion.keras"
PREDICTOR_NAME = f"CCE_Conv2D_{DATASET}.keras"
N_IMAGES = 1000
N_TRIALS = 100
N_VALIDATION_IMAGES = 1000
N_VALIDATION_RUNS = 3
TARGET_METRIC = "acc_both"  # Maximizar el SSIM medio de las estimaciones.
SEPARATION_ITERATIONS = 20
assert TARGET_METRIC in {"acc_both", "ssim", "recon_bpsnr"}

# Pesos físicos de la mezcla: fijos para comparar las mismas imágenes.
FIXED_PARAMS = {"alpha_mix": 0.5}
DEFAULT_PARAMS = {
    "alpha_1": -2., "alpha_2": -22., "beta": 1.,
    "bias": 0.22, "slope": 22., "gamma": 0., "delta": 0.353,
    "saturation_object_threshold": 0.05, "saturation_window": 5,
    "confidence_projection": True, "confidence_scale": 0.1, "confidence_floor": 0.1,
    }


def suggest_params(trial):
    params = {
        **DEFAULT_PARAMS, **FIXED_PARAMS,
        "bias": trial.suggest_float("bias", 0.05, 0.45),
        "slope": trial.suggest_float("slope", 5., 60., log=True),
        "gamma": trial.suggest_float("gamma", 0., 2.),
        "delta": trial.suggest_float("delta", 0., 1.),
        "saturation_object_threshold": trial.suggest_float("saturation_object_threshold", 0., 0.2),
        "saturation_window": trial.suggest_categorical("saturation_window", [3, 5, 7, 9]),
        "confidence_projection": trial.suggest_categorical("confidence_projection", [True, False]),
    }
    if params["confidence_projection"]:
        params["confidence_scale"] = trial.suggest_float("confidence_scale", 0.01, 0.5, log=True)
        params["confidence_floor"] = trial.suggest_float("confidence_floor", 0.01, 0.5, log=True)
    params["alpha_1"] = trial.suggest_float("alpha_1", -30., 0.)
    params["alpha_2"] = trial.suggest_float("alpha_2", -30., 0.)
    params["beta"] = trial.suggest_float("beta", 0.8, 1.1)
    return params


tf.keras.utils.set_random_seed(SEED)
data = Loader.data(dataset=DATASET)
cvae = Loader.cvae(model_name=CVAE_NAME)
predictor = Loader.predictor(model_name=PREDICTOR_NAME)

# Generador independiente de las semillas del CVAE y de Optuna.
rng = np.random.default_rng(SEED)
idx0 = rng.permutation(len(data["x_train"]))[:N_IMAGES]
idx1 = rng.permutation(len(data["x_train_1"]))[:N_IMAGES]
train_batch = (
    data["x_train"][idx0], data["x_train_1"][idx1],
    data["y_train"][idx0], data["y_train_1"][idx1],
)


def scalar_metric(metrics, key):
    value = metrics[key]
    # Las métricas de calidad devuelven (media, desvío).
    if isinstance(value, (tuple, list)):
        value = value[0]
    return float(value)


def objective(trial):
    params = suggest_params(trial)
    tf.keras.utils.set_random_seed(SEED)
    model = crop3_test3_module.Crop3Test3(cvae=cvae, predictor=predictor)
    metrics = model.unmix(
        *train_batch,
        params=params,
        iterations=SEPARATION_ITERATIONS,
        labels=data["labels"],
        show_image=False,
        show_metrics=False,
    )
    score = scalar_metric(metrics, TARGET_METRIC)
    if not np.isfinite(score):
        raise ValueError(f"Métrica no finita: {TARGET_METRIC}={score}")
    trial.set_user_attr("effective_params", params)
    trial.set_user_attr("iterations_run", model.iterations_run)
    for key in ("acc_both", "ssim", "recon_bpsnr"):
        trial.set_user_attr(key, scalar_metric(metrics, key))
    return score


## Estudio

Búsqueda condicional: los parámetros de confianza se ajustan solo si la proyección está ponderada. Siempre se ajustan alpha/beta y se ejecutan todas las iteraciones. La primera prueba usa la configuración inicial. Cada prueba y sus parámetros efectivos se guardan en SQLite.


In [2]:
from datetime import datetime

output_dir = project_root / "project" / "experiments" / "optim_params" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
study_path = output_dir / f"crop3_test3_{DATASET}_{TARGET_METRIC}_{run_id}.sqlite3"
study = optuna.create_study(
    direction="maximize",
    storage=f"sqlite:///{study_path}",
    study_name=f"Crop3Test3_{DATASET}_{TARGET_METRIC}",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
print("Estudio persistente:", study_path, flush=True)
study.enqueue_trial(DEFAULT_PARAMS.copy())  # Referencia con la configuración inicial.
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
best_params = study.best_trial.user_attrs["effective_params"].copy()
print("Mejor valor:", study.best_value)
print("Parámetros completos:", best_params)


[I 2026-11-16 19:47:01,410] A new study created in RDB with name: Crop3Test3_fashion_acc_both


Estudio persistente: /home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/results/crop3_test3_fashion_acc_both_20261116_194700.sqlite3


[I 2026-11-16 19:47:05,865] Trial 0 finished with value: 0.715 and parameters: {'bias': 0.22, 'slope': 22.0, 'gamma': 0.0, 'delta': 0.353, 'saturation_object_threshold': 0.05, 'saturation_window': 5, 'confidence_projection': True, 'deterministic_latent': False, 'early_stop_tolerance': 0.0001, 'confidence_scale': 0.1, 'confidence_floor': 0.1, 'alpha_1': -2.0, 'alpha_2': -22.0, 'beta': 1.0, 'early_stop_patience': 3, 'min_iterations': 5}. Best is trial 0 with value: 0.715.
[I 2026-11-16 19:47:10,228] Trial 1 finished with value: 0.469 and parameters: {'bias': 0.3506302428817915, 'slope': 6.559518055743659, 'gamma': 0.9722425581148704, 'delta': 0.4998311777480342, 'saturation_object_threshold': 0.13142199980396865, 'saturation_window': 9, 'confidence_projection': False, 'deterministic_latent': True, 'early_stop_tolerance': 0.005, 'early_stop_patience': 2, 'min_iterations': 7}. Best is trial 0 with value: 0.715.
[I 2026-11-16 19:47:14,082] Trial 2 finished with value: 0.428 and parameters: 

Mejor valor: 0.74
Parámetros completos: {'alpha_1': -20.266128727763707, 'alpha_2': -5.283467251002054, 'beta': 0.8419790068002895, 'bias': 0.17981392741657354, 'slope': 50.39353297623054, 'gamma': 0.0005634132451439275, 'delta': 0.456798042420513, 'saturation_object_threshold': 0.057551592632187565, 'saturation_window': 5, 'confidence_projection': True, 'confidence_scale': 0.13867691618590477, 'confidence_floor': 0.06753488141559513, 'deterministic_latent': False, 'early_stop_tolerance': 0.001, 'early_stop_patience': 3, 'min_iterations': 5, 'alpha_mix': 0.5}


## Visualización de la optimización

Gráficos con Matplotlib: no requieren Plotly ni instalaciones dentro del notebook.

In [3]:
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
fig, ax = plt.subplots(figsize=(9, 4))
numbers = [t.number for t in completed]
values = np.array([t.value for t in completed])
ax.scatter(numbers, values, label="Prueba")
ax.plot(numbers, np.maximum.accumulate(values), label="Mejor hasta ahora", color="tab:orange")
ax.set_xlabel("Prueba")
ax.set_ylabel(TARGET_METRIC)
ax.grid(alpha=0.3)
ax.legend()
plt.show()
plt.close(fig)

names = sorted({name for t in completed for name in t.params})
columns = 3
rows = (len(names) + columns - 1) // columns
fig, axes = plt.subplots(rows, columns, figsize=(14, 3 * rows), squeeze=False)
for ax, name in zip(axes.flat, names):
    relevant = [t for t in completed if name in t.params]
    ax.scatter([t.params[name] for t in relevant], [t.value for t in relevant], alpha=0.7)
    ax.set_xlabel(name)
    ax.set_ylabel(TARGET_METRIC)
    ax.grid(alpha=0.3)
for ax in list(axes.flat)[len(names):]:
    ax.axis("off")
fig.tight_layout()
plt.show()
plt.close(fig)


/home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/optim_params_crop3_test3.ipynb:cell_5:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
/home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/optim_params_crop3_test3.ipynb:cell_5:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Validación de Crop3Test3

Evalúa los mejores parámetros en tres mezclas de test independientes; muestra curvas e imágenes de la primera corrida.


In [ ]:
best_params = study.best_trial.user_attrs["effective_params"].copy()
validation_results = []
validation_rng = np.random.default_rng(SEED + 1000)
for run in range(N_VALIDATION_RUNS):
    idx0 = validation_rng.permutation(len(data["x_test"]))[:N_VALIDATION_IMAGES]
    idx1 = validation_rng.permutation(len(data["x_test_1"]))[:N_VALIDATION_IMAGES]
    tf.keras.utils.set_random_seed(SEED + 1000 + run)
    validation_model = crop3_test3_module.Crop3Test3(cvae=cvae, predictor=predictor)
    metrics = validation_model.unmix(
        data["x_test"][idx0], data["x_test_1"][idx1],
        data["y_test"][idx0], data["y_test_1"][idx1],
        params=best_params.copy(),
        iterations=SEPARATION_ITERATIONS,
        labels=data["labels"],
        # Mostrar una grilla de 1000 imagenes seria ilegible. Las imagenes
        # de la primera corrida se muestran abajo con un subconjunto.
        show_image=False,
        show_metrics=False,
        show_all_curves=(run == 0),
    )
    validation_results.append({
        "run": run + 1,
        "iterations_run": validation_model.iterations_run,
        **{key: scalar_metric(metrics, key) for key in ("acc_both", "ssim", "recon_bpsnr")},
    })
    print(validation_results[-1])
    if run == 0:
        first_validation_model = validation_model
        first_validation_indices = (idx0, idx1)

# Mostrar solo 10 mezclas de la primera corrida, con sus labels finales.
from project.helper.graphics.graphics import Graphics
m = first_validation_model
idx0, idx1 = first_validation_indices
limit = min(10, len(idx0), len(idx1))
Graphics.complete_plot(
    tf.convert_to_tensor(m.mixed_input[:limit]),
    tf.convert_to_tensor(data["x_test"][idx0[:limit]]),
    tf.convert_to_tensor(data["x_test_1"][idx1[:limit]]),
    data["y_test"][idx0[:limit]], data["y_test_1"][idx1[:limit]],
    tf.convert_to_tensor(m.source1_estimation[:limit]),
    tf.convert_to_tensor(m.source2_estimation[:limit]),
    tf.convert_to_tensor(m.mask1[:limit]), tf.convert_to_tensor(m.mask2[:limit]),
    m.predictions1[:limit], m.predictions2[:limit],
    tf.convert_to_tensor(m.source1_estimation[:limit]),
    model_params=best_params,
    class_labels=data["labels"],
)


## Resultados de validación

In [5]:
validation_summary = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, key in zip(axes, ("acc_both", "ssim", "recon_bpsnr")):
    values = np.array([row[key] for row in validation_results])
    validation_summary[key] = {"mean": float(values.mean()), "std": float(values.std())}
    ax.plot(np.arange(1, len(values) + 1), values, "o-")
    ax.axhline(values.mean(), color="tab:orange", linestyle="--")
    ax.set_title(f"{key}: {values.mean():.4f} ± {values.std():.4f}")
    ax.set_xlabel("Corrida de validación")
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)
print(validation_summary)


/home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/optim_params_crop3_test3.ipynb:cell_9:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
{'acc_both': {'mean': 0.6943333333333334, 'std': 0.015151090903151315}, 'ssim': {'mean': 0.7629970908164978, 'std': 0.0028225340551717862}, 'recon_bpsnr': {'mean': 21.259730021158855, 'std': 0.08133415294341959}}


## Guardar estudio y resultados

Guarda parámetros completos, todas las pruebas y validación en un archivo pickle; no guarda batches de imágenes.

In [6]:
import pickle
from datetime import datetime

output_dir = project_root / "project" / "experiments" / "optim_params" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"crop3_test3_{DATASET}_{TARGET_METRIC}_{datetime.now():%Y%m%d_%H%M%S}.pkl"
with output_path.open("wb") as stream:
    pickle.dump({
        "model": "Crop3Test3",
        "dataset": DATASET,
        "cvae": CVAE_NAME,
        "predictor": PREDICTOR_NAME,
        "seed": SEED,
        "objective": TARGET_METRIC,
        "best_params": best_params,
        "study": study,
        "validation_results": validation_results,
        "validation_summary": validation_summary,
    }, stream)
print("Guardado:", output_path)


Guardado: /home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/results/crop3_test3_fashion_ssim_20261116_193113.pkl
